# 향후 희망 여가활동 Class Imbalance 개선 실험
- 2024~2025년 향후 희망 여가활동 1순위 예측의 class imbalance 문제를 점검함.
- 다항 로지스틱 기반 class weight 강도 조절 실험을 수행함.
- 전체 정확도와 소수 중분류 Recall / F1 사이의 trade-off를 비교함.

## 분석 환경 및 데이터 경로 설정
- 분석에 필요한 패키지와 경로를 설정함.
- 현재 위치와 무관하게 프로젝트 루트를 탐색하도록 설정함.

In [ ]:
import pathlib
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    log_loss,
    f1_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    recall_score,
)
from sklearn.utils.class_weight import compute_class_weight

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

BASE_PATH = pathlib.Path().resolve()

PROJECT_PATH = None
for path in [BASE_PATH, *BASE_PATH.parents]:
    if (path / "notebooks" / "preference" / "data").exists():
        PROJECT_PATH = path
        break

if PROJECT_PATH is None:
    raise FileNotFoundError("???? ??? ?? ?????.")

PREFERENCE_PATH = PROJECT_PATH / "notebooks" / "preference"
SOURCE_PATH = PREFERENCE_PATH / "data" / "source"
PROCESSED_PATH = PREFERENCE_PATH / "data" / "processed" / "satisfaction"

SURVEY_PATH = SOURCE_PATH / "leisure_activity_survey_2021_2025_selected_columns.csv"
MAPPING_PATH = PROCESSED_PATH / "ml_activity_category_mapping.csv"

print("PROJECT_PATH:", PROJECT_PATH)
print("SURVEY_PATH 존재:", SURVEY_PATH.exists())
print("MAPPING_PATH 존재:", MAPPING_PATH.exists())

## 데이터 불러오기 및 향후 희망 순위 테이블 생성
- 2024~2025년 향후 희망 여가활동 응답만 사용함.
- 기존 문화누리 중분류 매핑표를 그대로 적용함.
- 학습 제외 분류와 중복 중분류를 제거한 뒤 유효 중분류 순위를 재구성함.

In [ ]:
survey = pd.read_csv(SURVEY_PATH, encoding="utf-8-sig", low_memory=False)
activity_mapping = pd.read_csv(MAPPING_PATH, encoding="utf-8-sig")

activity_mapping["활동코드"] = pd.to_numeric(
    activity_mapping["활동코드"],
    errors="coerce",
).astype("Int64")

future_rank_cols_raw = [
    "향후 희망하는 여가활동 1순위",
    "향후 희망하는 여가활동 2순위",
    "향후 희망하는 여가활동 3순위",
]

valid_categories = (
    activity_mapping
    .loc[activity_mapping["학습타깃사용여부"], "중분류"]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

feature_cols = [
    "조사년도",
    "성별",
    "연령",
    "17개 시도",
    "지역규모",
    "최종가중치",
]

mapping_info = (
    activity_mapping
    .set_index("활동코드")[["여가활동명", "중분류", "학습타깃사용여부"]]
    .to_dict("index")
)

survey_2425 = survey[survey["조사년도"].isin([2024, 2025])].copy()
future_base = survey_2425[feature_cols + future_rank_cols_raw].copy().reset_index(drop=True)
future_base["응답자_ID"] = [
    f"PREF_{i:05d}"
    for i in range(1, len(future_base) + 1)
]

rank_rows = []

for idx, row in future_base.iterrows():
    valid_rank = []
    exclude_count = 0
    duplicate_count = 0
    
    for col in future_rank_cols_raw:
        code = pd.to_numeric(row[col], errors="coerce")
        
        if pd.isna(code):
            continue
        
        code = int(code)
        info = mapping_info.get(code)
        
        if info is None:
            exclude_count += 1
            continue
        
        category = info["중분류"]
        
        if not bool(info["학습타깃사용여부"]):
            exclude_count += 1
            continue
        
        if category in valid_rank:
            duplicate_count += 1
            continue
        
        valid_rank.append(category)
    
    temp = {
        "응답자_ID": row["응답자_ID"],
        "향후희망_유효순위수": len(valid_rank),
        "제외분류_제거수": exclude_count,
        "중복중분류_제거수": duplicate_count,
    }
    
    for rank in range(3):
        temp[f"향후희망_유효중분류_{rank + 1}순위"] = valid_rank[rank] if rank < len(valid_rank) else np.nan
    
    rank_rows.append(temp)

future_rank_info = pd.DataFrame(rank_rows)
future_rank_base = future_base.drop(columns=future_rank_cols_raw).merge(
    future_rank_info,
    on="응답자_ID",
    how="left",
)

rank_cols = [
    "향후희망_유효중분류_1순위",
    "향후희망_유효중분류_2순위",
    "향후희망_유효중분류_3순위",
]

print("원 응답자 수:", len(future_base))
print("학습 가능 응답자 수:", future_rank_base[rank_cols[0]].notna().sum())
print("유효순위수 분포")
print(future_rank_base["향후희망_유효순위수"].value_counts(dropna=False).sort_index())
print("제외분류 제거수 합:", future_rank_base["제외분류_제거수"].sum())
print("중복중분류 제거수 합:", future_rank_base["중복중분류_제거수"].sum())

## 입력 변수 라벨 생성 및 모델 데이터 확정
- 성별, 연령대, 시도, 지역규모 라벨을 생성함.
- 1순위 유효 중분류가 있는 응답자만 학습 대상으로 사용함.
- 직접 대응 입력변수 결측을 확인함.

In [ ]:
sex_map = {
    1: "남성",
    2: "여성",
}

age_map = {
    1: "15-19세",
    2: "20대",
    3: "30대",
    4: "40대",
    5: "50대",
    6: "60대",
    7: "70세 이상",
}

sido_map = {
    1: "서울",
    2: "부산",
    3: "대구",
    4: "인천",
    5: "광주",
    6: "대전",
    7: "울산",
    8: "세종",
    9: "경기",
    10: "강원",
    11: "충북",
    12: "충남",
    13: "전북",
    14: "전남",
    15: "경북",
    16: "경남",
    17: "제주",
}

region_size_map = {
    1: "대도시",
    2: "중소도시",
    3: "읍면지역",
}

model_df = future_rank_base[future_rank_base[rank_cols[0]].notna()].copy()

model_df["성별_라벨"] = model_df["성별"].map(sex_map)
model_df["연령대"] = model_df["연령"].map(age_map)
model_df["시도"] = model_df["17개 시도"].map(sido_map)
model_df["지역규모_라벨"] = model_df["지역규모"].map(region_size_map)
model_df["조사년도_라벨"] = model_df["조사년도"].astype(str)

model_feature_cols = [
    "성별_라벨",
    "연령대",
    "시도",
    "지역규모_라벨",
    "조사년도_라벨",
]

target_col = rank_cols[0]

model_missing = (
    model_df[model_feature_cols + [target_col, "최종가중치"]]
    .isna()
    .sum()
    .reset_index(name="결측치")
    .rename(columns={"index": "칼럼명"})
)
model_missing["결측률"] = model_missing["결측치"] / len(model_df)

display(model_missing)

print("1순위 중분류 분포")
display(
    model_df[target_col]
    .value_counts(normalize=True)
    .rename_axis("중분류")
    .reset_index(name="비율")
)

## Train / Valid / Test 분리
- 70:15:15 비율로 데이터를 분리함.
- 조사년도와 1순위 중분류 분포가 유지되도록 stratify를 적용함.

In [ ]:
split_key = (
    model_df["조사년도_라벨"]
    + "_"
    + model_df[target_col]
)

train_valid_idx, test_idx = train_test_split(
    model_df.index,
    test_size=0.15,
    random_state=42,
    stratify=split_key,
)

train_valid_df = model_df.loc[train_valid_idx].copy()
train_valid_key = (
    train_valid_df["조사년도_라벨"]
    + "_"
    + train_valid_df[target_col]
)

train_idx, valid_idx = train_test_split(
    train_valid_df.index,
    test_size=0.15 / 0.85,
    random_state=42,
    stratify=train_valid_key,
)

train_df = model_df.loc[train_idx].copy()
valid_df = model_df.loc[valid_idx].copy()
test_df = model_df.loc[test_idx].copy()

split_summary = pd.DataFrame({
    "데이터": ["train", "valid", "test"],
    "응답자수": [len(train_df), len(valid_df), len(test_df)],
    "비율": [len(train_df) / len(model_df), len(valid_df) / len(model_df), len(test_df) / len(model_df)],
})

display(split_summary)

display(pd.crosstab(
    pd.concat([
        train_df.assign(split="train"),
        valid_df.assign(split="valid"),
        test_df.assign(split="test"),
    ])[target_col],
    pd.concat([
        train_df.assign(split="train"),
        valid_df.assign(split="valid"),
        test_df.assign(split="test"),
    ])["split"],
    normalize="columns",
))

## 모델 입력 배열 및 평가 함수 생성
- 범주형 입력변수를 더미 변수로 변환함.
- 최종가중치를 평균 1로 정규화함.
- 전체 성능과 중분류별 성능을 평가하는 함수를 생성함.

In [ ]:
feature_df = pd.get_dummies(
    model_df[model_feature_cols],
    drop_first=True,
    dtype=float,
)

feature_columns = feature_df.columns.tolist()

category_to_idx = {
    category: idx
    for idx, category in enumerate(valid_categories)
}

idx_to_category = {
    idx: category
    for category, idx in category_to_idx.items()
}

X_train = feature_df.loc[train_df.index].to_numpy(dtype=float)
X_valid = feature_df.loc[valid_df.index].to_numpy(dtype=float)
X_test = feature_df.loc[test_df.index].to_numpy(dtype=float)

y_train = train_df[target_col].to_numpy()
y_valid = valid_df[target_col].to_numpy()
y_test = test_df[target_col].to_numpy()

def normalized_weight(data):
    sample_weight = data["최종가중치"].to_numpy(dtype=float)
    return sample_weight / np.nanmean(sample_weight)

weight_train = normalized_weight(train_df)
weight_valid = normalized_weight(valid_df)
weight_test = normalized_weight(test_df)

minority_categories = (
    train_df[target_col]
    .value_counts()
    .sort_values()
    .head(5)
    .index
    .tolist()
)

print("입력 더미 변수 수:", len(feature_columns))
print("소수 분류:", minority_categories)


def align_prob(model, X):
    raw_prob = model.predict_proba(X)
    prob = np.zeros((X.shape[0], len(valid_categories)))
    class_to_col = {
        category: idx
        for idx, category in enumerate(model.classes_)
    }
    
    for j, category in enumerate(valid_categories):
        if category in class_to_col:
            prob[:, j] = raw_prob[:, class_to_col[category]]
    
    row_sum = prob.sum(axis=1, keepdims=True)
    prob = np.divide(
        prob,
        row_sum,
        out=np.zeros_like(prob),
        where=row_sum > 0,
    )
    
    return prob


def evaluate_model(model_name, split_name, prob, y_true):
    pred_idx = np.argmax(prob, axis=1)
    pred_label = np.array([idx_to_category[idx] for idx in pred_idx])
    
    return {
        "모델": model_name,
        "데이터": split_name,
        "Top1_Accuracy": np.mean(pred_label == y_true),
        "Rank1_LogLoss": log_loss(y_true, prob, labels=valid_categories),
        "Macro_F1": f1_score(y_true, pred_label, labels=valid_categories, average="macro", zero_division=0),
        "Weighted_F1": f1_score(y_true, pred_label, labels=valid_categories, average="weighted", zero_division=0),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, pred_label),
        "소수분류_Recall": recall_score(
            y_true,
            pred_label,
            labels=minority_categories,
            average="macro",
            zero_division=0,
        ),
    }


def make_category_metric(model_name, split_name, prob, y_true):
    pred_idx = np.argmax(prob, axis=1)
    pred_label = np.array([idx_to_category[idx] for idx in pred_idx])
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        pred_label,
        labels=valid_categories,
        zero_division=0,
    )
    pred_count = (
        pd.Series(pred_label)
        .value_counts()
        .reindex(valid_categories, fill_value=0)
        .to_numpy()
    )
    
    return pd.DataFrame({
        "모델": model_name,
        "데이터": split_name,
        "중분류": valid_categories,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "실제건수": support,
        "예측건수": pred_count,
        "소수분류여부": [category in minority_categories for category in valid_categories],
    })

## Baseline 및 Class Weight 강도 조절 실험
- alpha별 class weight를 적용한 다항 로지스틱을 학습함.
- adjusted_weight = balanced_weight ** alpha 공식을 사용함.
- alpha가 커질수록 소수 분류 보정 강도가 커짐.

In [ ]:
alpha_list = [
    0.00,
    0.10,
    0.20,
    0.25,
    0.30,
    0.40,
    0.50,
    0.60,
    0.75,
    1.00,
]

base_class_weight_values = compute_class_weight(
    class_weight="balanced",
    classes=np.array(valid_categories),
    y=y_train,
)

base_class_weight = {
    category: weight
    for category, weight in zip(valid_categories, base_class_weight_values)
}

alpha_model_dict = {}
alpha_performance_rows = []
alpha_category_metric_list = []

for alpha in alpha_list:
    adjusted_class_weight = {
        category: base_class_weight[category] ** alpha
        for category in valid_categories
    }
    
    alpha_sample_weight = weight_train * pd.Series(y_train).map(adjusted_class_weight).to_numpy()
    alpha_sample_weight = alpha_sample_weight / np.nanmean(alpha_sample_weight)
    
    model_name = f"alpha_{alpha:.2f}"
    model = LogisticRegression(
        solver="lbfgs",
        max_iter=1000,
        C=1.0,
    )
    
    model.fit(
        X_train,
        y_train,
        sample_weight=alpha_sample_weight,
    )
    
    alpha_model_dict[alpha] = model
    print(model_name, "수렴 반복 횟수:", model.n_iter_[0])
    
    for split_name, X_split, y_split in [
        ("valid", X_valid, y_valid),
        ("test", X_test, y_test),
    ]:
        prob = align_prob(model, X_split)
        alpha_performance_rows.append(
            evaluate_model(model_name, split_name, prob, y_split)
        )
        alpha_category_metric_list.append(
            make_category_metric(model_name, split_name, prob, y_split)
        )

alpha_performance = pd.DataFrame(alpha_performance_rows)
alpha_category_metric = pd.concat(alpha_category_metric_list, ignore_index=True)

base_reference = (
    alpha_performance[alpha_performance["모델"] == "alpha_0.00"]
    [["데이터", "Top1_Accuracy", "Macro_F1", "Rank1_LogLoss", "소수분류_Recall"]]
    .rename(columns={
        "Top1_Accuracy": "기본_Top1_Accuracy",
        "Macro_F1": "기본_Macro_F1",
        "Rank1_LogLoss": "기본_Rank1_LogLoss",
        "소수분류_Recall": "기본_소수분류_Recall",
    })
)

alpha_performance_compare = alpha_performance.merge(
    base_reference,
    on="데이터",
    how="left",
)

alpha_performance_compare["기본대비_Top1_변화"] = alpha_performance_compare["Top1_Accuracy"] - alpha_performance_compare["기본_Top1_Accuracy"]
alpha_performance_compare["기본대비_Macro_F1_변화"] = alpha_performance_compare["Macro_F1"] - alpha_performance_compare["기본_Macro_F1"]
alpha_performance_compare["기본대비_LogLoss_변화"] = alpha_performance_compare["Rank1_LogLoss"] - alpha_performance_compare["기본_Rank1_LogLoss"]
alpha_performance_compare["기본대비_소수분류_Recall_변화"] = alpha_performance_compare["소수분류_Recall"] - alpha_performance_compare["기본_소수분류_Recall"]

alpha_performance_compare = alpha_performance_compare.round(4)

display(alpha_performance_compare)

print("valid 성능")
display(
    alpha_performance_compare[alpha_performance_compare["데이터"] == "valid"]
    .sort_values("Macro_F1", ascending=False)
)

print("test 성능")
display(
    alpha_performance_compare[alpha_performance_compare["데이터"] == "test"]
    .sort_values("Macro_F1", ascending=False)
)

## 후보 Alpha 선정 및 중분류별 성능 확인
- 기본 모델 대비 Top1 하락폭이 3%p 이내인 후보를 확인함.
- Macro F1과 소수 분류 Recall이 개선된 후보를 선택함.
- 선택 alpha의 test 중분류별 Recall / F1을 확인함.

In [ ]:
valid_alpha_compare = alpha_performance_compare[
    alpha_performance_compare["데이터"] == "valid"
].copy()

valid_alpha_compare["후보조건"] = (
    (valid_alpha_compare["기본대비_Top1_변화"] >= -0.03)
    & (valid_alpha_compare["기본대비_Macro_F1_변화"] > 0)
    & (valid_alpha_compare["기본대비_소수분류_Recall_변화"] > 0)
    & (valid_alpha_compare["기본대비_LogLoss_변화"] <= 0.20)
)

candidate_alpha_table = (
    valid_alpha_compare[valid_alpha_compare["후보조건"]]
    .sort_values(["Macro_F1", "소수분류_Recall"], ascending=False)
)

print("후보 alpha")
display(candidate_alpha_table)

if len(candidate_alpha_table) > 0:
    selected_alpha_name = candidate_alpha_table.iloc[0]["모델"]
else:
    selected_alpha_name = (
        valid_alpha_compare
        .sort_values("Macro_F1", ascending=False)
        .iloc[0]["모델"]
    )

selected_alpha = float(selected_alpha_name.replace("alpha_", ""))

print("선택 alpha:", selected_alpha)
display(alpha_performance_compare[alpha_performance_compare["모델"] == selected_alpha_name])

base_vs_selected_category = alpha_category_metric[
    (alpha_category_metric["데이터"] == "test")
    & (alpha_category_metric["모델"].isin(["alpha_0.00", selected_alpha_name]))
].copy()

base_vs_selected_category = base_vs_selected_category.sort_values([
    "중분류",
    "모델",
]).round(4)

display(base_vs_selected_category)